# Training on AMD

Trains the crack / no-crack patch classifier and the UNet crack segmenter with the repository scripts
(`holdpoint/vision/train_cls.py`, `holdpoint/vision/train_seg.py`). The scripts are device-agnostic: on a ROCm build
of PyTorch, `torch.cuda.is_available()` is true on AMD Instinct GPUs and `--device auto` picks the GPU.

Checkpoints stay in `runs/` (not committed; published as release assets). `results/train-<stamp>/` gets the
metrics JSON, training logs, checkpoint SHA-256 and the `amd-smi` dump, and is committed.

Install once per notebook session: `pip install -r requirements.txt segmentation-models-pytorch` (keep the
notebook's own ROCm torch; do not install the CPU wheels from `requirements-ml.txt`).

In [ ]:
import os, sys, json, shutil, subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
from holdpoint.vision import common
DEV = common.pick_device("auto")
INFO = common.device_info(DEV)
print(json.dumps(INFO, indent=2))
# amd-smi is the AMD tool; its output goes next to the results as evidence of the hardware
smi = shutil.which("amd-smi")
SMI = subprocess.run([smi, "static", "--asic", "--json"], capture_output=True, text=True).stdout if smi else None
print(SMI[:800] if SMI else "amd-smi not found")
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")

## Data
Downloads are checked against the SHA-256 published by Mendeley. SDNET2018's host asks for a browser check:
upload the zip from https://doi.org/10.15142/T3TD19 and set `SDNET_ZIP`; without it the classifier trains on
Özgenel's classification set.

In [ ]:
SDNET_ZIP = os.environ.get("SDNET_ZIP")  # path to SDNET2018.zip uploaded by hand, or None
!python data/download.py ozgenel_seg
!python data/download.py ozgenel_cls
if SDNET_ZIP:
    !python data/download.py sdnet2018 --from-file "$SDNET_ZIP"
    !python data/split.py sdnet2018
CLS_DATA = "sdnet2018" if SDNET_ZIP else "ozgenel_cls"
PSEUDO = CLS_DATA
print("classifier data:", CLS_DATA)

## Classifier

In [ ]:
CLS_OUT = REPO / "runs" / f"cls-{STAMP}"
cls_metrics = __import__("holdpoint.vision.train_cls", fromlist=["main"]).main([
    "--dataset", CLS_DATA, "--arch", os.environ.get("HOLDPOINT_CLS_ARCH", "resnet18"),
    "--epochs", os.environ.get("HOLDPOINT_CLS_EPOCHS", "5"), "--batch", "128", "--workers", "8",
    "--amp", "--device", "auto", "--out", str(CLS_OUT)])
print(json.dumps(cls_metrics["test"], indent=2))

## Segmentation (real masks + pseudo masks)

In [ ]:
SEG_OUT = REPO / "runs" / f"seg-{STAMP}"
seg_metrics = __import__("holdpoint.vision.train_seg", fromlist=["main"]).main([
    "--pseudo", PSEUDO, "--pseudo-limit", "2000", "--epochs", os.environ.get("HOLDPOINT_SEG_EPOCHS", "30"),
    "--batch", "16", "--size", "384", "--amp", "--device", "auto", "--out", str(SEG_OUT)])
print(json.dumps(seg_metrics["test"], indent=2))

## Save evidence to results/

In [ ]:
RES = REPO / "results" / f"train-{STAMP}"
RES.mkdir(parents=True, exist_ok=True)
for name, d in (("cls", CLS_OUT), ("seg", SEG_OUT)):
    for f in ("metrics.json", "train.log", "model.pt.sha256"):
        shutil.copy(d / f, RES / f"{name}-{f}")
if SMI:
    (RES / "amd-smi.json").write_text(SMI)
(RES / "device.json").write_text(json.dumps(INFO, indent=2))
print(sorted(p.name for p in RES.iterdir()))